# PyTorch Quickstart

## Libraries

`torch`: Core library, provides tensors similar to NumPy arrays but with automatic differentiation.

`torch.nn`: Provides predefined layers (e.g. `nn.Linear`), activation functions, and loss functions.

`torch.optim`: Provides optimization algorithms like `SGD` or `Adam`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

## Tensors

### Creating tensors and tensor properties 

In [ ]:
# Basics

# Create tensor from a list, array, tuple, scalar...
x = torch.tensor([1,2,3])
arr = np.array([1,2,3])
x = torch.tensor(arr) # create a copy of arr (new memory location) 

# Alternative way to create tensor from array:
x = torch.from_numpy(arr) # shares memory with the input; preserves numpy dtype

# as_tensor: shares memory with the input; especially efficient for large arrays
x = torch.as_tensor([1,2,3])

################################################################################

# Creete tensor filled with specific values
x = torch.zeros(2,3) # 2x3 tensor of 0s
x = torch.ones(5) # 1d tensor of 1s
x = torch.rand(2,3) # 2x3 tensor, values 0 to 1, uniform
x = torch.randn(2,3) # 2x3 tensor, values drawn from norm. dist. (mean 0, std 1)
x = torch.randint(2, 6, (3,)) # random integers from 2 to 6, e.g. tensor([4, 3, 4])  
x = torch.arange(5)  # tensor([ 0,  1,  2,  3,  4])
x = torch.arange(1, 4) # tensor([ 1,  2,  3])

# Create empty tensor
x = torch.empty(2,3) # 2x3 tensor

# Create a new tensor with the same properties (shape, dtype, device)
x = torch.zeros_like(x)
x = torch.ones_like(x)
x = torch.empty_like(x)

################################################################################

# Tensor configurations (dtype, device, requires_grad)
x = torch.rand(2,3, dtype=torch.float32, device="cpu", requires_grad=True)

# create a copy of your tensor to a device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = x.to(device)
# If you already know GPU exists:
x = x.cuda()
# NOTE: model and data must be on the same device. See the section below.

# By default, requires_grad = False (i.e. not trainable)
# Turn this on for parameters to be trained: Every tensor with `requires_grad=True` tracks its operations.
# Turn this off for other tensors (e.g. input tensor, labels, etc.)
# Turn this off if you want to freeze trainable parameters

# During evaluation (or inference), you also want to turn this off for the
# trainable parameters, but it's recommended to use the torch.no_grad() method
# so you don't need to worry about forgetting to turning it back on
with torch.no_grad(x):
    # do something
    x


### Changing shapes and joining tensors

In [ ]:
# Note: Row vector and column vector cannot be distinguished using tensors.
# Their tensors are 1D and have the same shape.
# For example, a row or column vector of 3 elements can be a tensor of size 3,
# but not 1x3 or 3x1 (these are 2D, not 1D!)

################################################################################

# Reshaping: torch.reshape() or x.reshape()
a = torch.arange(4.)
torch.reshape(a, (2, 2)) # convert to 2x2 shape: tensor([[ 0.,  1.], [ 2.,  3.]])
b = torch.tensor([[0, 1], [2, 3]]) 
torch.reshape(b, (-1,))  # remove one dimension: tensor([ 0,  1,  2,  3])

# Transpose two specified dimensions: torch.transpose() or x.transpose()
x.transpose(0,1) # swap rows and columns

# Permute dimensions
x = torch.randn(2, 3, 5) # 2x3x5 tensor
torch.permute(x, (2, 0, 1)) # 5x2x3 tensor

################################################################################

# Unsqueeze: add a dimension of size 1 at specified index of dimension
x = torch.tensor([1, 2, 3]) # shape is [3]
torch.unsqueeze(x, 0) # add to dimension 0: tensor( [[1,   2,   3]] ) (new shape is [1,3])
torch.unsqueeze(x, 1) # add to dimension 1: tensor( [[1], [2], [3]] ) (new shape is [3, 1])
                      # Example use case: NN layers expect data in batches. If you only have one vector, 
                      # you can "fake" a batch of size 1 using x.unsqueeze(0).
                    

# Squeeze: remove specified dimensions of size 1
## Note: Global squeeze, x.squeeze(), remove ALL dimensions of size 1. Use with caution.
## Inplace squeeze (memory efficient): x.squeeze_() (notice the underscore)
x = torch.zeros(2, 1, 2, 1, 2) # tensor of shape [2, 1, 2, 1, 2]
y = torch.squeeze(x) # remove all dimensions of size 1. Get tensor of shape [2,2,2]
y = torch.squeeze(x, 0) # remove dimension 0 if the size is 1. (No change)
y = torch.squeeze(x, 1) # remove dimension 1 if the size is 1. Get tensor of shape [2, 2, 1, 2]
y = torch.squeeze(x, (1, 2, 3)) # remove dimensions 1, 2, and 3 if the size is 1. Get tensor of shape [2, 2, 2]

## Example use case: Clean up output tensor for downstream operations. For example,
## you have predictions for a batch of 4 samples, [[0.6], [0.2], [0.5], [0.9]].
## You want to compute the loss against the true labels: [1, 0, 1, 1].
## Squeeze the raw output to get [0.6, 0.2, 0.5, 0.9] for computing the loss.

################################################################################
    
# Slicing and joining
torch.cat((x, x, x), 0) # concatenate a list of tensors along an existing dimension
torch.stack((x, x), dim=1) # stack tensors along a new dimension
torch.split(x, split_size_or_sections=[1,4], dim=1) # split tensor into chunks of equal or specified sizes


x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
torch.cat((x, x, x), dim=0) # Combines tensors along dim 0 (vertical). Resulting shape: (6, 3)
torch.stack((x, x), dim=1) # stack tensors along a new dimension (Inserts a new dimension at index 1). Resulting shape: (2, 2, 3)
torch.split(x, split_size_or_sections=[1, 2], dim=1) # Split into specified chunk sizes
                                                     # Here: Splits dim 1 (columns) into sections of 1 and 2. Total columns must equal 3.
                                                     # Returns a tuple of two tensors of shape (2, 1) and (2, 2)


### Math operations and reduction

In [ ]:
# Element-wise operations
x = a - b
x = a + b
x = a / b
x = a * b

# Matrix multiplication
x = torch.matmul(a, b)
x = a @ b 

# In-place operation (works for any of the above operations)
# Since no copy will be created, this is a faster approach
# Just add an underscore to the method name, e.g.:
a.add_(b)

# Sum, mean, min, max ("Reduction")
# For a 2D tensor, dim=1 means operation along the column (return row results)
torch.sum(a, 1) # return row sums (e.g. if a is 3x5, output is a 1D tensor of size 3)
torch.min(a, 1) # return 2 tensors: row min values AND their indices

## Example - a simple linear model

### Preparing training data

In [ ]:
# Get device type
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Simple linear relationship: y = 2x
x_train = torch.tensor([[1.0], [2.0], [3.0]], device=device)
y_train = torch.tensor([[2.0], [4.0], [6.0]], device=device)

# Note: Tensors must be `float32` for neural network computations.

### Defining the model

In Pytorch, a **model** is a class (object) that keep track of its **states** (**parameters**).

To define a model, you will need to define:
- its structure (**neural network layers**) and 
- how inputs are processed (**forward method**).

In [ ]:
class MyModel(nn.Module):

    # Define the layers
    def __init__(self, input_dim, output_dim):
        super().__init__() 
        self.linear = nn.Linear(input_dim, output_dim)
    
    # Override the default forward method
    def forward(self, x):
        return self.linear(x)

Notes:

- Why does Pytorch represent a model as a class instead of a function? Because a class can store states (parameters) and methods together. This allows:
    - easier tracking of parameters and gradients
    - composition of layers and reuse
- `class LinearModel(nn.Module)`: Inherit from `nn.Module` so PyTorch can track parameters.
- `__init__`: Constructor called when instantiating the model.
- `super().__init__()`: 
    - Initializes the parent `nn.Module` class to register parameters internally.
    - For Python 2 compatibility, use this: `super(MyModel, self).__init__()` instead.
- `forward(self, x)`: Called when you run `model(x)`.
- *Notice the capital letters in nn.Module and nn.Linear!*

### Instantiate and move model to specified device

Model and data MUST be on the same device.

In [ ]:
input_dim = 1
output_dim = 1
model = MyModel(input_dim, output_dim).to(device) # In-place move (only works for models, not tensors)

### Training loop

Major steps: 
- **Forward Pass**: Input → model → prediction → loss.
- **Backward Pass (backpropagation)**: Computes gradients for all parameters.
- **Optimizer Step**: Updates parameters using gradients.

Loss functions:
- Regression: `nn.MSELoss()`, `nn.L1Loss()`
- Binary classification: `nn.BCELoss()`, `nn.BCEWithLogitsLoss()`
- Multi-class classification: `nn.CrossEntropyLoss()`

Optimizers:
- SGD: basic gradient descent, slower.
- Adam: adaptive learning rates, faster convergence in most cases.


In [ ]:
# 1. Setup Loss and Optimizer
criterion = nn.MSELoss()  # Computes Mean Squared Error Loss for regression
optimizer = optim.SGD(model.parameters(), lr=0.01)

Notes:
- `criterion`: Compute loss and returns a tensor (`loss` in the example) with `requires_grad=True` (so gradients can be computed when `loss.backward()` is called)
- `model.parameters()`: Retrieves tensors of *learnable parameters* (tensors with `requires_grad=True`), e.g.:
    - `model.linear.weight`
    - `model.linear.bias`
    - (and parameters in other layers if present)
- `optimizer`: Updates model parameters using gradients stored in `.grad` (e.g. `model.linear.weight.grad`).
- The whole computation process, from computing loss to updating parameters, will be explained later.

In [ ]:
# 2. The Loop
for epoch in range(100):
    model.train() # Set to training mode 

    # Forward Pass (prediction + compute loss)
    y_pred = model(x_train)
    loss = criterion(y_pred, y_train)
    
    # Backward Pass (Autograd)
    optimizer.zero_grad() # Clear old gradients from previous step
    loss.backward()       # Compute new gradients
    optimizer.step()      # Update model parameters

    # Print progress
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")


Notes:
- `model.train()`: Enables training mode for layers like dropout and batchnorm. If you don’t use these layers, switching modes is optional, but still good practice.
- `optimizer.zero_grad()`: Resets gradients to zero. 
    - PyTorch accumulates gradients by default (`param.grad += new_grad`), which is useful for gradient accumulation across mini-batches. However, we don't need that in this simple example.
- `y_pred`: contains tensors with `requires_grad=True`.
- `loss`: A tensor, not just a numerical value! It remembers how it was computed:
    - The `loss` tensor stores a reference to the function that created it (actually the backward counterpart for the loss function `criterion`) in `.grad_fn`, where the tensors used in that calculation (e.g. `y_pred`) are embedded.
    - `y_pred` itself stores its computation history (model operations), this allows `loss.backward()` to trace back from `loss` to model parameters and calculate their gradients.
- ` loss.backward()`: Traverses computation graph backward (`loss` > `y_pred` > model parameters), computes gradients using the chain rule, and stores them in `.grad` of each parameter (e.g. `model.linear.weight.grad`).
- `optimizer.step()`: Reads `.grad` from parameters and updates `.data` (e.g. `model.linear.weight.data`).

Computation graph: 

Autograd: When you run `model(x_train)`, PyTorch builds a DAG (Directed Acyclic Graph), where each tensor knows its parents and the operation used to create it.

```mermaid
flowchart LR
    %% Styles
    classDef tensor fill:#D5F5E3,stroke:#27AE60,stroke-width:2px,color:#1B4F72,font-weight:bold,rx:12,ry:12
    classDef tensor_no_grad fill:#D0D0D0,stroke:#7F8C8D,stroke-width:2px,color:#2C3E50,font-weight:bold,rx:12,ry:12
    classDef op fill:#FCF3CF,stroke:#F1C40F,stroke-width:2px,color:#7D6608,font-weight:bold,rx:8,ry:8

    %% Input Tensors
    X("<code>x</code>")
    Y_true("<code>y_true</code>")

    %% Model Parameter
    W("<code>model.linear.weight</code>")

    %% Forward Pass
    LinearOp(["Linear Layer"])
    Y_pred("<code>y_pred</code>")
    LossOp(["Compute Loss"])
    Loss("<code>loss</code>")

    %% Backward Pass
    BackwardOp(["Compute Gradients"])
    Grad_W("<code>model.linear.weight.grad</code>")

    %% Optimizer Step
    OptimStep(["Optimizer Step"])
    Updated_W("<code>model.linear.weight (updated)</code>")

    %% Connections - Forward
    X --> LinearOp
    W --> LinearOp
    LinearOp --> Y_pred
    Y_pred --> LossOp
    Y_true --> LossOp
    LossOp --> Loss

    %% Connections - Backward
    Loss --> BackwardOp
    BackwardOp --> Grad_W

    %% Connections - Optimizer
    Grad_W --> OptimStep
    W --> OptimStep
    OptimStep --> Updated_W

    %% Classes
    class X,Y_true,Y_pred tensor_no_grad
    class W,Grad_W,Loss,Updated_W tensor
    class LinearOp,LossOp,BackwardOp,OptimStep op
```


### Evaluation

In [ ]:
model.eval() # Set to evaluation mode (affects Dropout/BatchNorm)

with torch.no_grad(): # Disable gradient tracking for efficiency
    predicted = model(x_train)
    mse = criterion(predicted, y_train)
    print(f'Final MSE on training data: {mse.item():.4f}')

## Training a classifier on MNIST

- This is a **multi-class classification** task (classifying digits 0 to 9, so 10 classes in total).
- We assume that **classes are mutually exclusive**, i.e. each input belong to only one class.



### Prepare the data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader # Handles batching, shuffling, parallel loading
from torchvision import datasets, transforms # datasets (MNIST, CIFAR) and Image preprocessing utilities

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create data preprocessing function
transform = transforms.Compose([
    transforms.ToTensor(), # converts PIL image to tensor of values [0,1]
    transforms.Normalize((0.1307,), (0.3081,)) # Standardizes data (mean, std of MNIST)
])                                             # Normalization improves training stability and convergence

# Downloads MNIST (Format: (image, label) pairs)
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform # Applies transforms on-the-fly
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# DataLoaders (separate loaders for training and test sets)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,   # 64 images per mini-batch
    shuffle=True,    # stochastic training
    num_workers=2,   # parallel data loading
    pin_memory=True  # faster CPU to GPU transfer
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

#### Why do we use a `DataLoader`?

- Training on the entire dataset at once is inefficient and often impossible: Neural networks are trained on chunks of data, not the full dataset. Loading all 60,000 MNIST images at once wastes memory and prevents efficient GPU parallelism.
- `DataLoader` allows for:
  - **Batching**: splits the dataset into manageable batches automatically.
  - **Shuffling** (`shuffle=True`):
    - prevents the model from learning order-specific patterns (without shuffling, gradient descent can perform poorly because consecutive batches may be correlated or ordered by class)
    - reduces bias
    - improves convergence
  - **Parallel data loading** (`num_workers > 0`): Worker processes load and preprocess data in parallel with model computation. This overlaps CPU preprocessing with GPU computation and helps prevent the GPU from sitting idle.
  - **Device-friendly memory handling** (`pin_memory=True`): Pinned CPU memory can make CPU-to-GPU transfers faster and enables non-blocking transfers. (e.g. `images = images.to(device, non_blocking=True)`)

#### What is a minibatch?

- **minibatch**: a small subset of the training data used to compute one gradient update.
- Attention: “SGD” in PyTorch almost never means `batch_size=1`. It *usually means mini-batch gradient descent* with an optimizer such as `torch.optim.SGD`.
    | Method | Batch size |
    | --- | ---: |
    | Batch Gradient Descent | Entire dataset |
    | Mini-batch Gradient Descent | 16-1024 samples |
    | Stochastic Gradient Descent | 1 sample |

- Modern deep learning almost always uses mini-batches because they provide a good tradeoff between:
  - **Statistical efficiency:** gradients are more stable than with one sample.
  - **Computational efficiency:** GPUs can process many samples in parallel.
  - **Generalization:** gradient noise can act as a useful form of regularization.
- **Almost all PyTorch operations assume batched input**. Even for one image, PyTorch expects a batch-shaped tensor:
    ```text
    (1, C, H, W)  # One image with C channels and a size of H x W
    ```

### Define the model

In [ ]:
# Model
model = nn.Sequential(
    nn.Flatten(),  # Converts Input (N, 1, 28, 28) to (N, 784)
    nn.Linear(28 * 28, 256), # Fully connected, learns global pixel patterns
    nn.ReLU(), # Introduces non-linearity
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10) # Outputs 10 logits (raw scores), one per class (0–9)
).to(device)

Notes:
- Instead of `nn.Module`, we use `nn.Sequential` here: 
    - Great for simple feedforward models:
        $x \rightarrow f_1 \rightarrow f_2 \rightarrow f_3 \rightarrow y$
    - Ideal when the model has no branching, no skip connections, no multiple inputs/outputs
    - Pros: Concise, readable, fewer bugs, automatically registers parameters
- Input dimensions: `(N, 1, 28, 28)`:
    - N: batch size, i.e. number of images in the minibatch (e.g. 64)
    - 1: number of channels is 1 for MNIST (grayscale image)
    - 28, 28: H x W pixels
- Since linear layers expect 1D vector for each sample, we first flatten the input to a 1D vector.
- We then use three fully-connected layers to learn the pixel patterns.
    - First linear layer learns from 784 pixels and outputs 256 features.
    - Second linear layer learns from 256 features outputs 128 features.
    - Last linear layer learns from 128 features and outputs 10 logits (raw scores).
- In between linear layers, we use ReLU to introduces non-linearity.
- Why no softmax layer? 
    - `nn.CrossEntropyLoss()` internally applies `LogSoftmax` and `NegativeLogLikelihood`
    - Only do softmax during loss computation to avoid training on softmax values (probability-like) for numerically stablility.


### Training and Evaluation

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()

    total_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1) # get class index from logits (raw scores)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Notes:
- One epoch is one training loop over the entire dataset, not one batch of data.
- `optim.Adam`: Adaptive learning rate. Faster convergence than SGD.
- `(predictions == labels).sum().item()`: `.item()` converts scalar tensor to number

## Training a CNN on MNIST

A bit more advanced and efficient training which involves:
- Custom Dataset
- GPU + mixed precision
- Logging
- Checkpointing & resume
- Clean training / evaluation abstraction

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torch.cuda.amp import autocast, GradScaler

BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3
CHECKPOINT_PATH = "checkpoint.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.1307,), (0.3081,))])

# Custom Dataset
# Multiple data sources, on-the-fly augmentation, label preprocessing
class MNISTDataset(Dataset):
    def __init__(self, train=True, transform=None):
        self.dataset = datasets.MNIST(
            root="./data",
            train=train,
            download=True,
            transform=transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx): 
        # returns 1 sample, deterministic, executed in parallel by workers
        image, label = self.dataset[idx]
        return image, label

# DataLoaders

train_loader = DataLoader(MNISTDataset(train=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

test_loader = DataLoader(MNISTDataset(train=False, transform=transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

### Define the model

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Layers to learn/extract features
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # In: (N, 1, 28, 28) Out: (N, 32, 28, 28). Padding preserves spatial size.
            nn.ReLU(),
            nn.MaxPool2d(2), # Downsamples by factor of 2: (N, 32, 14, 14)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # Output final feature map of (N, 64, 7, 7)
        )

        # Layers to learn classification using extracted features
        self.classifier = nn.Sequential(
            nn.Flatten(), # Output: 1D vector of 3136 (64 * 7 * 7) features
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10) # Final layer. Returns 10 logits
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
model = CNN().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scaler = GradScaler() # For Mixed Precision Training (AMP). See next section

### Load checkpoint to resume training

Checkpoints save model weights and optimizer states.

In [ ]:
start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch = checkpoint["epoch"] + 1

### Training and Evaluation

In [ ]:
# Training loop
for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        # Use AMP for efficiency (see next section)
        with autocast(): 
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {avg_loss:.4f}")

    # Save a checkpoint (model weights + optimizer state)
    torch.save({"epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict()
                }, CHECKPOINT_PATH)

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()
accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Notes:

**Non-blocking GPU transfers**

- `images.to(device, non_blocking=True)`
- Works with `pin_memory=True`, overlaps CPU-GPU transfer with computation.

**Mixed Precision Training (AMP)** 

- This uses FP16 where safe, only uses FP32 where necessary, reducing memory usage and increasing throughput.
- `from torch.cuda.amp import autocast, GradScaler`: For AMP, we need `autocast` and `GradScaler`.
    ```python
    scaler = GradScaler()
    with autocast():
        outputs = model(images)
        loss = criterion(outputs, labels)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    ```

- `with autocast()`: This automatically chooses precision per operation and avoids numerical instability
- `scaler.scale(loss).backward()`: FP16 gradients can underflow. Scaling prevents zero gradients.
- Use AMP on GPU ~ 20–50% speedup


## Efficiency and tricks

- Mixed Precision Training (AMP): see above
- Use `pin_memory=True` (GPU only): speeds up host to device transfers.
- Increase `num_workers`: Parallel data loading
    - `num_workers = min(8, os.cpu_count())`
- Try increase batch size until GPU is saturated (memory is your constraint).
- Avoid Python overhead in loops
- Pre-allocate tensors when possible
- Use vectorized operations
- Profile before optimizing, never guess bottlenecks:
    - `torch.profiler`
- Don’t prematurely optimize Python loops: Data loading is usually the bottleneck.
- Use `.eval()` and `torch.no_grad()` for faster inference speed.
- Enable non-blocking GPU transfers

## BatchNorm and Dropout

**Dropout**: Randomly zeroes activations during training. Disabled during evaluation. PyTorch handles this automatically if using `nn.Dropout`.

**BatchNorm**: Normalizes activations per mini-batch:

$$\hat{x} = \frac{x - \mu_\text{batch}}{\sqrt{\sigma_\text{batch}^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

**Training mode**: uses batch statistics, updates running averages.

**Evaluation mode**: uses running averages to normalize.